# Step 9 - Silver-label baseline classifier (explicit / implicit / neutral / none)

**Status: baseline on rule-derived (silver) labels. Not validated against human labels.** Every number here measures how well a model reproduces our own Step 6 rules, not whether it detects bias. When the Step 8 sample is annotated, the same pipeline is retrained and evaluated on human labels (section 8).

Target (unit level, from the Step 8 schema): `explicit` (stated gender restriction, "Male/Female" listing, age limit, civil-status requirement), `implicit` (coded wording or appearance/physical proxy), `neutral_statement` (explicit inclusion such as "any gender"), `none`. Priority when several apply: explicit > neutral_statement > implicit.

Models: TF-IDF + logistic regression, and Step 4 sentence embeddings + logistic regression. Evaluation: 5-fold cross-validation grouped by `employer_id`, so one employer's templates never sit on both sides. No transformer fine-tuning: with silver labels it would only memorise the rules.

Inputs: `step4_sentence_units.csv`, `step4_sentence_embeddings.npy`, `step6_sentence_level.csv`. Outputs: `outputs/equal_step9_silver_baseline/`.

## 1. Environment and reproducibility log

In [25]:
import json, os, platform, re, sys
from pathlib import Path

import numpy as np, pandas as pd
import sklearn, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, f1_score
from sklearn.model_selection import GroupKFold

SEED = 42
N_SPLITS = 5
OUT = Path("outputs") / "equal_step9_silver_baseline"
OUT.mkdir(parents=True, exist_ok=True)

env = {
    "python": sys.version.split()[0], "platform": platform.platform(), "processor": platform.processor(),
    "logical_cpus": os.cpu_count(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "seed": SEED, "cv": f"GroupKFold({N_SPLITS}) by employer_id",
}
try:
    import psutil
    env["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError:
    env["ram_gb"] = "not captured (psutil not installed)"
(OUT / "step9_environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
print(json.dumps(env, indent=2))

{
  "python": "3.12.9",
  "platform": "Windows-11-10.0.26200-SP0",
  "processor": "AMD64 Family 23 Model 113 Stepping 0, AuthenticAMD",
  "logical_cpus": 12,
  "numpy": "2.5.4",
  "pandas": "3.0.6",
  "scikit_learn": "1.9.1",
  "torch": "2.14.1+cpu",
  "cuda_available": false,
  "seed": 42,
  "cv": "GroupKFold(5) by employer_id",
  "ram_gb": 31.9
}


## 2. Load units and build silver labels

Word lists and regexes are copied from the Step 6 notebook (same Gaucher-style stems, same explicit and proxy patterns) so the silver labels match what Step 6 reports. Marital-status and age patterns now map to `explicit`, per the Step 8 schema; appearance maps to `implicit`.

In [26]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FALSE_POSITIVE_PREFIXES = ("confidential", "committee", "shareholder", "kinds")

def coded_words(text, stems):
    hits = []
    for w in re.findall(r"[a-z][a-z-]*", text.lower()):
        if w.startswith(FALSE_POSITIVE_PREFIXES):
            continue
        parts = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in parts for s in stems):
            hits.append(w)
    return hits

MALE_RE = re.compile(r"\b(?:male|males|men|gentlemen)\b", re.I)
FEMALE_RE = re.compile(r"\b(?:female|females|women|ladies)\b", re.I)
INCLUSIVE_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                          r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
MARITAL_RE = re.compile(r"\b(?:single|married|unmarried|widow(?:ed|er)?|marital|pregnan\w*)\b", re.I)
AGE_RE = re.compile(r"\b(?:\d{2}\s*(?:-|to)\s*\d{2}\s*(?:years|yrs|yo|y/o|years\s+old)|"
                    r"(?:below|under|not\s+(?:more|older)\s+than|up\s+to|maximum\s+(?:age\s+)?of)\s*\d{2}|"
                    r"age[sd]?\s*(?:range|limit|requirement)?\s*[:\-]?\s*\d{2})\b", re.I)
APPEAR_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable|attractive|"
                       r"height|weight|physically\s+fit|heavy\s+lifting|tall|slim)\b", re.I)

units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
emb = np.load("outputs/equal_step4_sentence_embeddings/step4_sentence_embeddings.npy")
flags = pd.read_csv("outputs/equal_step6_bias_analysis/step6_sentence_level.csv", encoding="utf-8-sig", dtype={"sentence_id": str})
flags["explicit_unit_type"] = flags["explicit_unit_type"].fillna("")
assert len(units) == len(emb) == len(flags) == 7971
df = units.merge(flags[["sentence_id", "explicit_unit_type", "masc_hits", "fem_hits"]], on="sentence_id", how="left", validate="one_to_one")
assert (df["sentence_id"].values == units["sentence_id"].values).all()

def silver(r):
    s = r["sentence"]
    t = r["explicit_unit_type"]
    if t in ("restrictive_male", "restrictive_female") or (t == "inclusive" and not INCLUSIVE_RE.search(s)) or MARITAL_RE.search(s) or AGE_RE.search(s):
        return "explicit"
    if t == "inclusive":
        return "neutral_statement"
    if int(r["masc_hits"]) + int(r["fem_hits"]) > 0 or APPEAR_RE.search(s):
        return "implicit"
    return "none"

df["silver"] = df.apply(silver, axis=1)
counts = df["silver"].value_counts()
display(counts.to_frame("units").assign(share=lambda d: (d.units / len(df)).round(4)))
print("employers:", df["employer_id"].nunique(), "| distinct unit texts:", df["sentence"].nunique())
print("distinct texts per class:"); print(df.groupby("silver")["sentence"].nunique().to_string())

,units,share
silver,,
none,6819,0.8555
implicit,668,0.0838
explicit,484,0.0607


employers: 320 | distinct unit texts: 4716
distinct texts per class:
silver
explicit      50
implicit     446
none        4220


## 3. Cross-validated baselines (employer-grouped)

`class_weight="balanced"` is the imbalance mitigation tested here. Metrics: macro-F1 (treats rare classes equally), per-class precision/recall/F1, and per-class PR-AUC (average precision) from out-of-fold probabilities.

In [27]:
CLASSES = [c for c in ["none", "implicit", "explicit", "neutral_statement"] if c in counts.index]
print("classes present:", CLASSES, "| absent:", [c for c in ["neutral_statement"] if c not in CLASSES])
y = df["silver"].map({c: i for i, c in enumerate(CLASSES)}).values
groups = df["employer_id"].values
folds = list(GroupKFold(n_splits=N_SPLITS).split(df, y, groups))

def lr(weighted=True):
    return LogisticRegression(max_iter=2000, class_weight="balanced" if weighted else None, random_state=SEED)

def run_cv(make_features, weighted=True):
    proba = np.zeros((len(df), len(CLASSES)))
    for tr, te in folds:
        Xtr, Xte = make_features(tr, te)
        m = lr(weighted).fit(Xtr, y[tr])
        p = m.predict_proba(Xte)
        proba[np.ix_(te, m.classes_)] = p
    return proba

def tfidf_features(texts):
    def f(tr, te):
        v = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        return v.fit_transform(texts[tr]), v.transform(texts[te])
    return f

def emb_features(tr, te):
    return emb[tr], emb[te]

def summarise(name, proba):
    pred = proba.argmax(1)
    rows = {"model": name, "macro_F1": round(f1_score(y, pred, average="macro"), 3)}
    for i, c in enumerate(CLASSES):
        rows[f"F1_{c}"] = round(f1_score(y == i, pred == i), 3)
        rows[f"PR_AUC_{c}"] = round(average_precision_score(y == i, proba[:, i]), 3)
    return rows

texts = df["sentence"].values
results, probas = [], {}
for name, feat, w in [("tfidf_lr_balanced", tfidf_features(texts), True), ("tfidf_lr_unweighted", tfidf_features(texts), False),
                      ("embedding_lr_balanced", emb_features, True), ("embedding_lr_unweighted", emb_features, False)]:
    probas[name] = run_cv(feat, w)
    results.append(summarise(name, probas[name]))
res = pd.DataFrame(results).set_index("model")
res.to_csv(OUT / "step9_cv_results.csv")
display(res)

classes present: ['none', 'implicit', 'explicit'] | absent: ['neutral_statement']


,macro_F1,F1_none,PR_AUC_none,F1_implicit,PR_AUC_implicit,F1_explicit,PR_AUC_explicit
model,,,,,,,
tfidf_lr_balanced,0.920,0.979,0.994,0.799,0.875,0.980,0.996
tfidf_lr_unweighted,0.888,0.977,0.995,0.723,0.866,0.965,0.996
embedding_lr_balanced,0.831,0.940,0.989,0.603,0.716,0.950,0.996
embedding_lr_unweighted,0.864,0.970,0.988,0.644,0.738,0.978,0.996


In [28]:
best = "tfidf_lr_balanced"
print(f"Per-class report, {best} (out-of-fold, silver labels)")
print(classification_report(y, probas[best].argmax(1), target_names=CLASSES, digits=3, zero_division=0))
print(f"Per-class report, embedding_lr_balanced")
print(classification_report(y, probas["embedding_lr_balanced"].argmax(1), target_names=CLASSES, digits=3, zero_division=0))

Per-class report, tfidf_lr_balanced (out-of-fold, silver labels)
              precision    recall  f1-score   support

        none      0.983     0.976     0.979      6819
    implicit      0.770     0.831     0.799       668
    explicit      0.985     0.975     0.980       484

    accuracy                          0.963      7971
   macro avg      0.913     0.927     0.920      7971
weighted avg      0.965     0.963     0.964      7971

Per-class report, embedding_lr_balanced
              precision    recall  f1-score   support

        none      0.983     0.901     0.940      6819
    implicit      0.470     0.843     0.603       668
    explicit      0.916     0.988     0.950       484

    accuracy                          0.901      7971
   macro avg      0.789     0.910     0.831      7971
weighted avg      0.936     0.901     0.912      7971



## 4. Keyword-removed ablation

The silver labels are defined by keywords, so a model can score well just by spotting them. Here every rule trigger (gender words, coded stems, marital/age/appearance patterns) is **deleted** before TF-IDF (a placeholder would still reveal that a trigger was present). The labels are unchanged. A large drop shows the model was relying on the rule triggers; what remains is context the rules did not capture (or noise).

In [29]:
STEM_RE = re.compile(r"\b(?:" + "|".join(re.escape(s) for s in MASC + FEM) + r")[a-z-]*", re.I)
NOUN_RE = re.compile(r"\b(?:man|boys?|woman|girls?|lady|gentleman)\b", re.I)

def strip_triggers(s):
    for rx in (MALE_RE, FEMALE_RE, INCLUSIVE_RE, MARITAL_RE, AGE_RE, APPEAR_RE, STEM_RE, NOUN_RE):
        s = rx.sub(" ", s)
    return s

stripped = np.array([strip_triggers(s) for s in df["sentence"]])
print("units changed by stripping:", int((stripped != texts).sum()))
proba_abl = run_cv(tfidf_features(stripped), True)
abl = pd.DataFrame([summarise("tfidf_lr_balanced_keywords_removed", proba_abl)]).set_index("model")
abl.to_csv(OUT / "step9_ablation_results.csv")
display(pd.concat([res.loc[["tfidf_lr_balanced"]], abl]))

units changed by stripping: 1173


,macro_F1,F1_none,PR_AUC_none,F1_implicit,PR_AUC_implicit,F1_explicit,PR_AUC_explicit
model,,,,,,,
tfidf_lr_balanced,0.920,0.979,0.994,0.799,0.875,0.980,0.996
tfidf_lr_balanced_keywords_removed,0.685,0.902,0.981,0.510,0.557,0.644,0.817


## 5. Does exact-text repetition inflate the scores?

3,255 unit texts are exact repeats. Employer-grouped folds already prevent one employer's text from crossing sides, but the same text can come from several employers. This check scores only the first occurrence of each distinct text.

In [30]:
first = ~df["sentence"].duplicated().values
rows = []
for name in ["tfidf_lr_balanced", "embedding_lr_balanced"]:
    p = probas[name][first]; yy = y[first]
    rows.append({"model": name, "units": int(first.sum()), "macro_F1_distinct_texts": round(f1_score(yy, p.argmax(1), average="macro"), 3)})
display(pd.DataFrame(rows).set_index("model"))

,units,macro_F1_distinct_texts
model,,
tfidf_lr_balanced,4716,0.876
embedding_lr_balanced,4716,0.780


## 6. Where the model disagrees with the silver labels

Out-of-fold errors of the TF-IDF model, for human review. These are the first units an annotator should look at: either the rule is wrong or the model found context.

In [31]:
pd.set_option("display.max_colwidth", 120)
pred = probas["tfidf_lr_balanced"].argmax(1)
err = df.assign(predicted=[CLASSES[i] for i in pred])[pred != y]
err[["sentence_id", "silver", "predicted", "sentence"]].to_csv(OUT / "step9_oof_disagreements.csv", index=False, encoding="utf-8-sig")
print("disagreements:", len(err), "of", len(df))
display(pd.crosstab(err["silver"], err["predicted"]))
display(err.groupby(["silver", "predicted"]).head(2)[["silver", "predicted", "sentence"]].sort_values(["silver", "predicted"]).reset_index(drop=True))

disagreements: 291 of 7971


predicted,explicit,implicit,none
silver,,,
explicit,0,7,5
implicit,0,0,113
none,7,159,0


,silver,predicted,sentence
0,explicit,implicit,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and one meter and fifty-seven centi...
1,explicit,implicit,"With pleasing personality, female and single"
2,explicit,none,HR Recruitment Staff (Preferably Male)
3,explicit,none,Married
4,implicit,none,Dependable and is willing to be given additional task when needed
5,implicit,none,"Polite, with professional disposition and ability to remain calm in stressful driving conditions"
6,none,explicit,20 Years old Above
7,none,explicit,25 Years old Above
8,none,implicit,"Ability to receive, stock and/or deliver and invoicing procedures"
9,none,implicit,Has the ability to easily develop good rapport with respondent.


## 7. Save the silver labels

Saved for reuse in Step 10. Marked as silver so they are never mistaken for ground truth.

In [32]:
df[["sentence_id", "record_id", "employer_id", "sentence", "silver"]].rename(columns={"silver": "silver_bias_type"}).to_csv(
    OUT / "step9_silver_labels.csv", index=False, encoding="utf-8-sig")
print(sorted(p.name for p in OUT.iterdir()))

['step9_ablation_results.csv', 'step9_cv_results.csv', 'step9_detection_summary.csv', 'step9_detection_units.csv', 'step9_environment.json', 'step9_oof_disagreements.csv', 'step9_silver_labels.csv', 'step9_silver_model.joblib']


## 8. Retraining on human labels (when Step 8 is annotated)

1. Fill the label columns of `step8_annotation_sample.csv` and run its validator cell.
2. Use the human `bias_type` as the target for the annotated units and keep the same employer-grouped folds.
3. Evaluate on human labels only: macro-F1, per-class PR-AUC, and the keyword-removed ablation.
4. Report the silver-label scores above only as the rule-reproduction baseline.

Limitations of this notebook: labels come from our own rules, so scores describe rule reproduction; the proxy and inclusive patterns are crude; no external corpus has been tested; CPU only.

## 9. Bias detection test (single ad or many ads)

**Status: silver-label baseline, not validated.** The detector fits the TF-IDF + logistic regression model on all 7,971 silver-labelled units and saves it to `step9_silver_model.joblib`.

For each ad it (1) masks emails, URLs and phone numbers, (2) splits the ad into one unit per line, removing bullets such as `~`, (3) scores each unit with the model, and (4) checks the same unit against the Step 6 / Step 8 rules to give the **attribute, direction and reason**.

Final label rule: if a rule fires, the rule label is used and the model is shown as a cross-check. If no rule fires but the model predicts a bias class, the unit is labelled `model only - review` with the terms that drove the prediction. Units where the rule and model disagree should be read by a human.

Limits: line-level units only, no span offsets, and the rules are crude (for example, `single` also matches non-marital uses).

In [33]:
import joblib

det_vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
det_clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED).fit(
    det_vec.fit_transform(df['sentence'].values), df['silver'].values)
joblib.dump({'vectorizer': det_vec, 'classifier': det_clf, 'trained_on': 'silver labels, not validated'}, OUT / 'step9_silver_model.joblib')
DET_CLASSES = list(det_clf.classes_)
print('detector classes:', DET_CLASSES)

EMAIL_RE = re.compile(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}')
URL_RE = re.compile(r'(?:https?://|www\.)\S+|\b[\w\-]+\.(?:com|ph|org|net|edu|gov)(?:/\S*)?\b', re.I)
PHONE_RE = re.compile(r'(?<!\d)(?:\+?63|0)?\d[\d\s\-()]{6,}\d(?!\d)')
BULLET_RE = re.compile(r'^\s*(?:[~\u2022\u25cf\u25aa\u25e6\u2023\u25cb\u25a0\u25a1\u25c6\u27a2\u27a4\u2713\u2714]+|[-*](?=\s|$))\s*')
PLACEHOLDER_RE = re.compile(r'\[(?:EMAIL|URL|PHONE)\]')

def mask_contacts(t):
    return PHONE_RE.sub('[PHONE]', URL_RE.sub('[URL]', EMAIL_RE.sub('[EMAIL]', t)))

def split_units(text):
    out = []
    for line in str(text).splitlines():
        line = BULLET_RE.sub('', mask_contacts(line)).strip()
        if not re.search(r'[A-Za-z]', PLACEHOLDER_RE.sub('', line)):
            continue
        parts = re.split(r'(?<=[a-z\)])\.\s+(?=[A-Z])', line) if len(line) > 160 else [line]
        out += [p.strip() for p in parts if p.strip()]
    return out

def rule_evidence(s):
    explicit, implicit, attrs = [], [], []
    direction, coding, neutral = 'na', 'na', None
    inc, m, f = INCLUSIVE_RE.search(s), MALE_RE.search(s), FEMALE_RE.search(s)
    if inc:
        neutral = f'explicit inclusion wording ({inc.group(0)!r})'
    elif m and f:
        explicit.append(f'lists both genders ({m.group(0)!r}, {f.group(0)!r}), limiting applicants to a stated gender')
        attrs.append('gender'); direction = 'both_listed'
    elif m or f:
        direction = 'male' if m else 'female'
        explicit.append(f'restricts applicants to {direction} ({(m or f).group(0)!r})')
        attrs.append('gender')
    for rx, attr, label in ((MARITAL_RE, 'marital_status', 'marital/civil-status requirement'), (AGE_RE, 'age', 'age limit')):
        k = rx.search(s)
        if k:
            explicit.append(f'{label} ({k.group(0)!r})'); attrs.append(attr)
    a = APPEAR_RE.search(s)
    if a:
        implicit.append(f'appearance/physical proxy ({a.group(0)!r})'); attrs.append('appearance_physical')
    mh, fh = coded_words(s, MASC), coded_words(s, FEM)
    if mh or fh:
        coding = 'agentic' if len(mh) > len(fh) else 'communal' if len(fh) > len(mh) else 'balanced'
        implicit.append(f'gender-coded wording ({coding}): ' + ', '.join(mh + fh)); attrs.append('gender')
    if explicit:
        return 'explicit', sorted(set(attrs)), direction, coding, explicit + implicit
    if neutral:
        return 'neutral_statement', ['gender'], direction, coding, [neutral]
    if implicit:
        return 'implicit', sorted(set(attrs)), direction, coding, implicit
    return 'none', [], direction, coding, []

def top_terms(s, class_name, n=3):
    x = det_vec.transform([s])
    contrib = x.multiply(det_clf.coef_[DET_CLASSES.index(class_name)]).tocoo()
    names = det_vec.get_feature_names_out()
    ranked = sorted(zip(contrib.data, contrib.col), reverse=True)[:n]
    return [names[c] for v, c in ranked if v > 0]

def detect_ad(text, ad_id='ad_1'):
    units_ = split_units(text)
    if not units_:
        return pd.DataFrame(columns=['ad_id', 'unit_no', 'text', 'bias_label', 'attribute', 'direction', 'gender_coding', 'model_label', 'model_confidence', 'source', 'why'])
    proba = det_clf.predict_proba(det_vec.transform(units_))
    rows = []
    for i, (u, pr) in enumerate(zip(units_, proba), 1):
        mlabel, mconf = DET_CLASSES[int(pr.argmax())], float(pr.max())
        rlabel, attrs, direction, coding, reasons = rule_evidence(u)
        if rlabel != 'none':
            final = rlabel
            source = 'rule + model agree' if mlabel == rlabel else f'rule (model said {mlabel})'
            why = '; '.join(reasons)
        elif mlabel != 'none':
            final, source = mlabel, 'model only - review'
            attrs = ['unspecified']
            why = 'no rule matched; model flagged it. Influential terms: ' + (', '.join(top_terms(u, mlabel)) or 'n/a')
        else:
            final, source, why = 'none', '-', ''
        rows.append({'ad_id': ad_id, 'unit_no': i, 'text': u, 'bias_label': final, 'attribute': ', '.join(attrs) or 'none',
                     'direction': direction, 'gender_coding': coding, 'model_label': mlabel, 'model_confidence': round(mconf, 2),
                     'source': source, 'why': why})
    return pd.DataFrame(rows)

def load_ads(ads, text_col='ad_text', id_col='ad_id'):
    if isinstance(ads, (str, Path)) and Path(str(ads)).suffix.lower() in ('.csv', '.xlsx'):
        p_ = Path(str(ads))
        t = pd.read_csv(p_, encoding='utf-8-sig', dtype=str) if p_.suffix.lower() == '.csv' else pd.read_excel(p_, dtype=str)
        ids = t[id_col] if id_col in t else pd.Series([f'ad_{i + 1}' for i in range(len(t))])
        return dict(zip(ids, t[text_col]))
    if isinstance(ads, dict):
        return ads
    return {f'ad_{i + 1}': a for i, a in enumerate(ads)}

def detect_ads(ads, **kw):
    ads = load_ads(ads, **kw)
    units_df = pd.concat([detect_ad(t, k) for k, t in ads.items()], ignore_index=True)
    def verdict(g):
        labels = set(g['bias_label'])
        v = 'EXPLICIT bias detected' if 'explicit' in labels else 'IMPLICIT bias detected' if 'implicit' in labels else 'no bias detected'
        flagged = g[g['bias_label'].isin(['explicit', 'implicit'])]
        return pd.Series({'units': len(g), 'explicit': int((g['bias_label'] == 'explicit').sum()), 'implicit': int((g['bias_label'] == 'implicit').sum()),
                          'neutral_statement': int((g['bias_label'] == 'neutral_statement').sum()), 'verdict': v,
                          'attributes': ', '.join(sorted(set(', '.join(flagged['attribute']).split(', ')) - {''})) or 'none'})
    summary = units_df.groupby('ad_id', sort=False).apply(verdict, include_groups=False).reset_index()
    return summary, units_df

def show_report(summary, units_df, show_all_units=False):
    pd.set_option('display.max_colwidth', 200); pd.set_option('display.width', 250)
    display(summary)
    shown = units_df if show_all_units else units_df[units_df['bias_label'] != 'none']
    if shown.empty:
        print('No flagged units.')
    else:
        display(shown[['ad_id', 'unit_no', 'text', 'bias_label', 'attribute', 'direction', 'model_confidence', 'source', 'why']].reset_index(drop=True))

detector classes: ['explicit', 'implicit', 'none']


### 9a. Single job post (test)

Paste one ad into `SINGLE_AD`. Set `SHOW_ALL_UNITS = True` to also list the units labelled `none`.

In [34]:
SINGLE_AD = '''Yamaha

Production Associates (70)

Qualifications:
~At least High School Graduate or 2-year Vocational Graduate or 2-year ~College Undergraduate
~With or without work experience
~Must be willing to render overtime

Interested applicants may send their resumes to:
~hrd_limarecruitment@yamaha-motor.com.ph
Head Office:
~Lots 1 & 2, Block 17, Phase I, LIMA Technology Center, Malvar, Batangas
~Tel. No.: +630434551900'''
SHOW_ALL_UNITS = False

single_summary, single_units = detect_ads({'single_test': SINGLE_AD})
show_report(single_summary, single_units, SHOW_ALL_UNITS)

,ad_id,units,explicit,implicit,neutral_statement,verdict,attributes
0,single_test,10,0,0,0,no bias detected,none


No flagged units.


### 9b. Multiple job ads (testing)

`MULTI_ADS` accepts a dict `{ad_id: text}`, a list of texts, or a path to a `.csv` / `.xlsx` file with an `ad_text` column (and an optional `ad_id` column), for example `MULTI_ADS = 'my_ads.xlsx'`. Results are saved to `step9_detection_summary.csv` and `step9_detection_units.csv`.

The third ad below is a made-up test case, not from the dataset.

In [35]:
MULTI_ADS = {
    'yamaha': SINGLE_AD,
    'puregold_hr_assistant': '''Puregold

Human Resource Assistant

Qualifications:
~Female/Male
~Graduate of BS Psychology, BSBA Human Resource Management, BS Behavioral Science or other related courses
~With at least 1 year experience in human resources functions
~With knowledge in implementing all human resources faucet

For inquiries and referral, visit or contact:
~web.facebook.com/LMDPESOMarinduqueOfficial
~lmdpeso@yahoo.com
~Mobile No.: 0912-673-0915''',
    'synthetic_test': '''Sales Associate

Qualifications:
~Single, 18-25 years old
~With pleasing personality
~Dependable and willing to be trained
~Strong leadership and competitive spirit''',
}

multi_summary, multi_units = detect_ads(MULTI_ADS)
multi_summary.to_csv(OUT / 'step9_detection_summary.csv', index=False, encoding='utf-8-sig')
multi_units.to_csv(OUT / 'step9_detection_units.csv', index=False, encoding='utf-8-sig')
show_report(multi_summary, multi_units)

,ad_id,units,explicit,implicit,neutral_statement,verdict,attributes
0,yamaha,10,0,0,0,no bias detected,none
1,puregold_hr_assistant,10,1,1,0,EXPLICIT bias detected,"gender, unspecified"
2,synthetic_test,6,1,3,0,EXPLICIT bias detected,"age, appearance_physical, gender, marital_status"


,ad_id,unit_no,text,bias_label,attribute,direction,model_confidence,source,why
0,puregold_hr_assistant,4,Female/Male,explicit,gender,both_listed,1.00,rule + model agree,"lists both genders ('Male', 'Female'), limiting applicants to a stated gender"
1,puregold_hr_assistant,8,"For inquiries and referral, visit or contact:",implicit,unspecified,na,0.66,model only - review,"no rule matched; model flagged it. Influential terms: and, for"
2,synthetic_test,3,"Single, 18-25 years old",explicit,"age, marital_status",na,0.98,rule + model agree,marital/civil-status requirement ('Single'); age limit ('18-25 years')
3,synthetic_test,4,With pleasing personality,implicit,appearance_physical,na,0.98,rule + model agree,appearance/physical proxy ('pleasing personality')
4,synthetic_test,5,Dependable and willing to be trained,implicit,gender,na,0.52,rule (model said none),gender-coded wording (communal): dependable
5,synthetic_test,6,Strong leadership and competitive spirit,implicit,gender,na,0.97,rule + model agree,"gender-coded wording (agentic): leadership, competitive"
